# GWS 2022 Analysis

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import plotly.express as px 
import statsmodels.formula.api as smf

: 

### Connecting to database

In [ ]:
db_path = "data/nba.sqlite"  # change to your path
conn = sqlite3.connect(db_path)

tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table';").fetchall()
print([t[0] for t in tables])

### Retreiving data
Getting both home and away game for the 2022 season

In [ ]:
query = """
SELECT *
FROM game as g
JOIN other_stats AS o ON g.game_id = o.game_id
JOIN game_info as i ON g.game_id = i.game_id
WHERE g.team_abbreviation_home LIKE "%GSW%"
AND g.season_id = "22021"
"""
df_home = pd.read_sql_query(query, conn)
df_home.head()

In [ ]:
query = """
SELECT *
FROM game AS g
JOIN other_stats AS o ON g.game_id = o.game_id
JOIN game_info as i ON g.game_id = i.game_id
WHERE matchup_home LIKE "%GSW%"
    AND g.season_id = "22021"
    AND g.team_abbreviation_home NOT LIKE "%GSW%"

"""
df_away = pd.read_sql_query(query, conn)
df_away.head()

In [ ]:
query = """
SELECT *
FROM game AS g
JOIN other_stats AS o ON g.game_id = o.game_id
JOIN game_info as i ON g.game_id = i.game_id
WHERE matchup_home  NOT LIKE "%GSW%"
    AND g.season_id = "22021"

"""
df_others = pd.read_sql_query(query, conn)
df_others.head()

### Removing duplicated columns and merging dfs

In [ ]:
df_home = df_home.loc[:, ~df_home.columns.duplicated()]
df_away = df_away.loc[:, ~df_away.columns.duplicated()]
df_others = df_others.loc[:, ~df_others.columns.duplicated()]

df_home["home"] = 1
df_away["home"] = 0 


df = pd.concat([df_home, df_away], ignore_index=True)
df.head()

In [ ]:
df.columns

### Exploring the performance of GWS

In [ ]:
df["win"] = np.where(
    ((df["team_abbreviation_home"] == "GSW") & (df["wl_home"] == "W")) |
    ((df["team_abbreviation_home"] != "GSW") & (df["wl_home"] == "L")),
    1,
    0,
 )

df_others["win"] = np.where(
    (df_others["wl_home"] == "W"), 1, 0 
)



In [ ]:
df.to_csv("data/clean_data/clean.csv")
df_others.to_csv("data/clean_data/clean_others.csv")

In [ ]:
df.groupby(by = "home").agg(
    win_total = ("win","sum"),
    total_games = ("home", "count")
)

In [ ]:
df["win_flag"] = df["win"].map({1: "1", 0: "0"})

fig = px.scatter(
    df,
    x="fg_pct_home",
    y="fg_pct_away",
    color="win_flag",
    color_discrete_map={
        "1": "#1f77b4",  # home/away win color
        "0": "#ff7f0e",  # loss color
    },
    category_orders={"win_flag": ["1", "0"]},
    labels={"win_flag": "win"},
)
fig.update_layout(xaxis_title="Home FG%", yaxis_title="Away FG%")
fig.show()

## Determinants of shot efficiency

In [ ]:
df.select_dtypes(include="number").columns

In [ ]:
gsw_home = df[df["team_abbreviation_home"] == "GSW"].copy()
gsw_home = gsw_home.select_dtypes(include="number")
gsw_home.columns

In [ ]:
gsw_home = gsw_home.drop(
    columns=["min", "home", "win", "video_available_home", "video_available_away", "pts_home", 
    "fgm_home", "fg3_pct_home", "fg3m_home"],
    errors="ignore",
)
corr = gsw_home.corr()


In [ ]:
fig = px.imshow(
    corr,
    text_auto=True,
    color_continuous_scale="RdBu",
    zmin=-1,
    zmax=1,
    title="Correlation Matrix"
)

fig.show()

In [ ]:

target = "fg_pct_home"


corr_with_target = (
    gsw_home.corr(numeric_only=True)[target]
    .drop(target)
    .sort_values(key=lambda s: s.abs(), ascending=False)
)

top_n = 15
corr_top = corr_with_target.head(top_n).reset_index()
corr_top.columns = ["feature", "corr"]

fig = px.bar(
    corr_top,
    x="corr",
    y="feature",
    orientation="h",
    title=f"Top {top_n} correlations with {target}",
)
fig.update_layout(template="plotly_dark")
fig.show()

In [ ]:
predictors = " + ".join(corr_top["feature"])
model = smf.ols(f"fg_pct_home ~ {predictors}", data=gsw_home)
results = model.fit()

In [ ]:
results.summary()